In [1]:
import pandas as pd 

In [ ]:
import pandas as pd
import re
import string

file_path = r"\Sentiment140\training.1600000.processed.noemoticon.csv"
columns = ['target', 'ids', 'date', 'flag', 'user', 'text']

df = pd.read_csv(file_path, encoding='latin-1', header=None, names=columns)


In [3]:

print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (1600000, 6)


,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [4]:
df['target'].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

In [5]:
df['target'] = df['target'].replace(4, 1)

In [6]:
print("Missing values:\n", df.isnull().sum())

Missing values:
 target    0
ids       0
date      0
flag      0
user      0
text      0
dtype: int64


In [7]:
print("Duplicates:\n", df.duplicated().sum())

Duplicates:
 0


In [8]:
df.duplicated().value_counts()

False    1600000
Name: count, dtype: int64

In [9]:
df[df['text'].str.contains(r'https?://\S+|www\.\S+', regex=True, na=False)][['text']].count()

text    75693
dtype: int64

In [10]:
df[df['text'].str.contains(r'@\w+', regex=True, na=False)][['text']].count()

text    738491
dtype: int64

In [11]:
df[df['text'].str.contains(r'#\w+', regex=True, na=False)][['text']].count()

text    35842
dtype: int64

In [12]:
# هيعرض الصفوف اللي فيها أي رمز مش حرف ولا رقم ولا مسافة (زي ! ؟ , " ' وغيرها)
df[df['text'].str.contains(r'[^\w\s]', regex=True, na=False)][['text']].count()

text    1449659
dtype: int64

In [13]:
# هيعرض الصفوف اللي بتحتوي على أي أرقام
df[df['text'].str.contains(r'\d+', regex=True, na=False)][['text']].count()

text    366487
dtype: int64

In [14]:
# هيعرض الصفوف اللي فيها مسافتين أو أكثر ورا بعض
df[df['text'].str.contains(r'\s{2,}', regex=True, na=False)][['text']].count()

text    564944
dtype: int64

In [15]:
df = df[['target', 'text']]


def clean_tweet(text):
    text = str(text).lower()                             # تحويل النص لحروف صغيرة
    text = re.sub(r'https?://\S+|www\.\S+', '', text)     # إزالة الروابط (URLs)
    text = re.sub(r'@\w+', '', text)                     # إزالة الإشارات للمستخدمين (@user)
    text = re.sub(r'#(\w+)', r'\1', text)                # الاحتفاظ بكلمة الهاشتاج وحذف علامة #
    text = re.sub(r'[^\w\s]', '', text)                  # إزالة علامات الترقيم والرموز الخاصة
    text = re.sub(r'\d+', '', text)                      # إزالة الأرقام
    text = re.sub(r'\s+', ' ', text).strip()             # إزالة المسافات الزائدة
    return text

df['clean_text'] = df['text'].apply(clean_tweet)

df[['text', 'clean_text', 'target']].head()

,text,clean_text,target
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww thats a bummer you shoulda got david carr...,0
1,is upset that he can't update his Facebook by ...,is upset that he cant update his facebook by t...,0
2,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...,0
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",no its not behaving at all im mad why am i her...,0


In [16]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)
negation_words = {'not', 'no', 'nor', 'neither', 'never', 'cannot', 'cant', 'wont', 'isnt', 'arent'}
stop_words = stop_words - negation_words

def remove_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w not in stop_words]
    return ' '.join(filtered)

df['text_no_stopwords'] = df['clean_text'].apply(remove_stopwords)
df[['clean_text', 'text_no_stopwords', 'target']].head()

,clean_text,text_no_stopwords,target
0,awww thats a bummer you shoulda got david carr...,awww thats bummer shoulda got david carr day d,0
1,is upset that he cant update his facebook by t...,upset cant update facebook texting result scho...,0
2,i dived many times for the ball managed to sav...,dived times ball managed save rest bounds,0
3,my whole body feels itchy and like its on fire,body feels itchy like,0
4,no its not behaving at all im mad why am i her...,no not behaving im mad cant,0


In [17]:
def stem_word(word):
    for suffix in ['ing', 'edly', 'fully', 'ed', 'ly', 'es', 's']:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[:-len(suffix)]
    return word

# Check stemming on sample words
sample_words = ['loving', 'loved', 'lovely', 'playing', 'crying', 'happiness']
for word in sample_words:
    print(f"{word:12} -> {stem_word(word)}")

def apply_stemming(text):
    words = text.split()
    stemmed = [stem_word(w) for w in words]
    return ' '.join(stemmed)

df['text_stemmed'] = df['text_no_stopwords'].apply(apply_stemming)
df[['text_no_stopwords', 'text_stemmed', 'target']].head()

loving       -> lov
loved        -> lov
lovely       -> love
playing      -> play
crying       -> cry
happiness    -> happines


,text_no_stopwords,text_stemmed,target
0,awww thats bummer shoulda got david carr day d,awww that bummer shoulda got david carr day d,0
1,upset cant update facebook texting result scho...,upset cant update facebook text result school ...,0
2,dived times ball managed save rest bounds,div tim ball manag save rest bound,0
3,body feels itchy like,body feel itchy like,0
4,no not behaving im mad cant,no not behav im mad cant,0


In [ ]:
from sklearn.model_selection import train_test_split

print("Target distribution:")
print(df['target'].value_counts())

X = df['text_stemmed']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training set: {len(X_train):,}")
print(f"Testing set:  {len(X_test):,}")

Target distribution:
target
0    800000
1    800000
Name: count, dtype: int64
Training set: 1,280,000
Testing set:  320,000


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=25000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Train shape:", X_train_tfidf.shape)
print("TF-IDF Test shape: ", X_test_tfidf.shape)

TF-IDF Train shape: (1280000, 25000)
TF-IDF Test shape:  (320000, 25000)


In [20]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

start_time = time.time()
lr_model = LogisticRegression(max_iter=1000, C=1.0)
lr_model.fit(X_train_tfidf, y_train)
ml_train_time = time.time() - start_time

y_pred_ml = lr_model.predict(X_test_tfidf)
ml_accuracy = accuracy_score(y_test, y_pred_ml)

print(f"Logistic Regression Accuracy: {ml_accuracy * 100:.2f}%")
print(f"Training Time: {ml_train_time:.2f} seconds")
print()
print(classification_report(y_test, y_pred_ml, target_names=['Negative (0)', 'Positive (1)']))

Logistic Regression Accuracy: 79.45%
Training Time: 3.63 seconds

              precision    recall  f1-score   support

Negative (0)       0.81      0.77      0.79    160000
Positive (1)       0.78      0.82      0.80    160000

    accuracy                           0.79    320000
   macro avg       0.80      0.79      0.79    320000
weighted avg       0.80      0.79      0.79    320000



In [29]:
# Test custom tweet with Machine Learning (TF-IDF + Logistic Regression)
input_text = "I absolutely love this new update! Everything is working smoothly."

clean_txt = clean_tweet(input_text)
nostop_txt = remove_stopwords(clean_txt)
stemmed_txt = apply_stemming(nostop_txt)

vectorized_input = tfidf.transform([stemmed_txt])
ml_pred = lr_model.predict(vectorized_input)[0]
ml_prob = lr_model.predict_proba(vectorized_input)[0]

predicted_label = "Positive (1) :)" if ml_pred == 1 else "Negative (0) :("
confidence = ml_prob[ml_pred] * 100

print("Original Text: ", input_text)
print("Preprocessed:  ", stemmed_txt)
print("Predicted:     ", predicted_label)
print(f"Confidence:    {confidence:.2f}%")

Original Text:  I absolutely love this new update! Everything is working smoothly.
Preprocessed:   absolute love new update work smooth
Predicted:      Positive (1) :)
Confidence:    90.45%
